# WaveGlow — Simplified Training Demo for visualization and debugging

This notebook provides an interactive training pipeline for the WaveGlow Neural Vocoder, mirroring the official `src/training/training-waveglow/train.py` logic but utilizing a live visual dashboard. 


In [1]:
import os
import sys
import json
import time
import glob
from pathlib import Path
from torch.utils.data import DataLoader
from IPython.display import display, clear_output
from tqdm import tqdm
import numpy as np
import scipy.stats as stats

import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "models" / "waveglow"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "data" / "loader_waveglow"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "training" / "training-waveglow"))

from glow import WaveGlow, WaveGlowLoss
from loader_waveglow import Mel2Samp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

In [2]:
# ==========================================
# 1. CONFIGURAÇÕES DO MODELO E DADOS
# ==========================================

waveglow_config = {
    "n_mel_channels": 80,
    "n_flows": 12,
    "n_group": 8,
    "n_early_every": 4,
    "n_early_size": 2,
    "WN_config": {
        "n_layers": 8,
        "n_channels": 256,
        "kernel_size": 3
    }
}

data_config = {
    "training_files": "data/processed/libriSpeech-en-tacotron-vae/mels_metadata.csv", # Ajuste para o dataset desejado
    "segment_length": 8000,
    "filter_length": 1024,
    "hop_length": 256,
    "win_length": 1024,
    "sampling_rate": 22050,
    "mel_fmin": 0.0,
    "mel_fmax": 8000.0
}

dataset = Mel2Samp(**data_config)
print(f"Dataset size: {len(dataset)}")

In [3]:
# ==========================================
# 2. INICIALIZAÇÃO DO MODELO
# ==========================================

model = WaveGlow(**waveglow_config).to(device)
criterion = WaveGlowLoss(sigma=1.0)

learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"WaveGlow Trainable Parameters: {num_params:,}")

In [4]:
# ==========================================
# 3. SETUP DE DIRETÓRIOS E CHECKPOINTS
# ==========================================
exp_dir = Path("notebooks/experiments/waveglow")
ckpt_dir = exp_dir / "checkpoints"
plots_dir = exp_dir / "plots"
ckpt_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

max_steps = 100000
save_interval = 500
batch_size = 8 # Batch reduzido por padrão devido ao alto uso de VRAM do WaveGlow
iteration = 0

loss_history = []

# Carregamento do Checkpoint Automático
latest_ckpt = max(glob.glob(str(ckpt_dir / "checkpoint_*.pt")), default=None, key=os.path.getctime)

if latest_ckpt:
    print(f"[*] Checkpoint encontrado! Retomando de: {latest_ckpt}")
    checkpoint = torch.load(latest_ckpt, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    iteration = checkpoint['iteration']
    loss_history = checkpoint['history']['loss']
    print(f"[*] Treino retomado a partir do passo {iteration}.")
else:
    print("[*] Nenhum checkpoint encontrado. Iniciando treino do zero.")

# ==========================================
# 4. SETUP DA INTERFACE GRÁFICA (2x2)
# ==========================================
plt.ion()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.tight_layout(pad=5.0)

# ==========================================
# 5. LOOP DE TREINAMENTO
# ==========================================
train_loader = DataLoader(dataset, batch_size=batch_size, num_workers=0, shuffle=True, drop_last=True)

model.train()
progress_bar = tqdm(total=max_steps, initial=iteration, desc="Treinando WaveGlow")

while iteration < max_steps:
    for batch in train_loader: 
        if iteration >= max_steps:
            break

        optimizer.zero_grad()
        
        mel, audio = batch
        mel = mel.to(device)
        audio = audio.to(device)
        
        # Forward pass (WaveGlow)
        outputs = model((mel, audio))

        # Cálculo da NLL (Negative Log-Likelihood)
        loss = criterion(outputs)
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

        progress_bar.set_postfix({'NLL Loss': f"{loss.item():.4f}"})
        progress_bar.update(1)

        # ==========================================
        # 6. DASHBOARD AO VIVO E ANÁLISE DO PRIOR (A cada 10 passos)
        # ==========================================
        if iteration % 10 == 0:
            z, log_s_list, log_det_W_list = outputs
            
            mel_target = mel[0].data.cpu().numpy()
            audio_target = audio[0].data.cpu().numpy()
            
            # O objetivo do Flow é mapear o áudio para um prior Gaussiano N(0, I)
            # Vamos extrair o z resultante para analisar a distribuição
            z_sample = z[0].detach().cpu().numpy().flatten()

            for ax in axes.flat:
                ax.clear()

            # [0, 0]: Mel Alvo
            axes[0, 0].imshow(mel_target, aspect='auto', origin='lower', cmap='magma')
            axes[0, 0].set_title(f"Condição: Mel-Spectrogram | Passo: {iteration}")
            axes[0, 0].set_ylabel("Mel Bins")
            axes[0, 0].set_xlabel("Frames")

            # [0, 1]: Áudio Alvo
            axes[0, 1].plot(audio_target, color='cyan', alpha=0.8)
            axes[0, 1].set_title("Alvo: Áudio Waveform")
            axes[0, 1].set_ylabel("Amplitude")
            axes[0, 1].set_xlabel("Amostras (Time)")
            axes[0, 1].grid(True, linestyle='--', alpha=0.5)

            # [1, 0]: NLL Loss
            axes[1, 0].plot(loss_history, color='red', alpha=0.8)
            axes[1, 0].set_title("WaveGlow NLL Loss")
            axes[1, 0].set_xlabel("Passos")
            axes[1, 0].set_ylabel("Loss")
            axes[1, 0].grid(True, linestyle='--', alpha=0.5)

            # [1, 1]: Distribuição do Latente Z vs Normal
            axes[1, 1].hist(z_sample, bins=50, density=True, alpha=0.6, color='green', label='Z (Flow Output)')
            
            # Plot da PDF da Normal Standard N(0, 1) para comparação
            xmin, xmax = axes[1, 1].get_xlim()
            x_pdf = np.linspace(xmin, xmax, 100)
            p_pdf = stats.norm.pdf(x_pdf, 0, 1)
            axes[1, 1].plot(x_pdf, p_pdf, 'k--', linewidth=2, label='N(0, 1) Prior')
            
            axes[1, 1].set_title("Diagnóstico Normalizing Flow (Latent Z)")
            axes[1, 1].set_xlabel("Valor Z")
            axes[1, 1].set_ylabel("Densidade")
            axes[1, 1].legend()

            clear_output(wait=True)
            display(fig)

        # ==========================================
        # 7. SALVAMENTO DE CHECKPOINT
        # ==========================================
        if iteration > 0 and iteration % save_interval == 0:
            ckpt_path = ckpt_dir / f"checkpoint_{iteration}.pt"
            plot_path = plots_dir / f"plot_{iteration}.png"
            
            torch.save({
                'iteration': iteration,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'waveglow_config': waveglow_config,
                'history': {
                    'loss': loss_history,
                }
            }, ckpt_path)
            
            fig.savefig(plot_path)

        iteration += 1

progress_bar.close()
plt.ioff()
plt.close(fig)
print("\n[✔] Treinamento interativo do WaveGlow finalizado!")
